# Activity 2: Interactive Charts with hvPlot

Yesterday every chart in this folder was a static image, a PNG you could zoom into with your own
eyes and nothing else. Today the same charts get a mouse: hover a bar to read its exact value, zoom
into a cluster of points, drag a dropdown to swap which slice of data you are looking at.

You will work with the same two datasets as Activity 1:

- `hospital_claims.parquet`: Medicare hospital billing data, one row per hospital per diagnosis group.
- `closing_price.csv`: daily closing stock prices for AAPL, MSFT, and IBM.

The notebook ends with a short look at Plotly Express, the charting library you will use again this
afternoon when you build a Streamlit app.

pandas already knows how to plot. The only thing missing is a second import. Import `pandas` as
usual, then import `hvplot.pandas`. That second import does not create a new object you call
directly, it registers a `.hvplot` accessor on every pandas DataFrame and Series that exists from
this point on. Nothing else about your code has to change: the data, the column names, and the
aggregations all stay exactly as they were. Expect no visible output, the cell just runs.

In [ ]:
import pandas as pd
import hvplot.pandas

Notebooks do not share state with each other, so this notebook starts from the same raw file
Activity 1 used. Load the claims data again. Expect no visible output yet, this just loads the
DataFrame into memory.

In [ ]:
claims = pd.read_parquet("../data/hospital_claims.parquet")

Rebuild the same ratio and state aggregate from Activity 1: charge to payment ratio, averaged by
state, highest first. Expect New Jersey, Nevada, and California at the top, all above 5.

In [ ]:
claims["ratio"] = claims["Average Covered Charges"] / claims["Average Total Payments"]
by_state = claims.groupby("Provider State")["ratio"].mean().sort_values(ascending=False)
by_state.head()

Here is yesterday's chart again, the plain static version with no `.hvplot` involved. Expect the
same unstyled bar chart as Activity 1: no title, no axis labels, cramped state codes, one flat
color. Keep it in mind, the next cell rebuilds it with one word changed.

In [ ]:
by_state.head(15).plot(kind="bar")

Change `.plot` to `.hvplot`. Same data, same `.head(15)`, same idea of a bar chart, nothing about
the DataFrame changed. Expect a bokeh chart with a toolbar in the corner: hover any bar to see its
exact state and ratio in a tooltip, and use the wheel zoom or pan tools to move around the chart.

In [ ]:
by_state.head(15).hvplot.bar()

hvPlot accepts the same idea of a title, axis labels, and a size, but the keyword names are its own,
not matplotlib's: `height` and `width` instead of `figsize`, and no `legend=False` needed for a
single series. Expect the same chart, now titled, colored, sized to 900 by 400, and rotated 45
degrees on the x-axis, still fully interactive.

In [ ]:
by_state.head(15).hvplot.bar(
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    ylabel="Average charge to payment ratio",
    xlabel="State",
    rot=45,
    height=400,
    width=900,
    color="#B31B1B",
)

A single bar chart proves the point, a multi-series time series proves it again. Load the closing
price data the same way you did in Activity 1: parse the `Date` column as dates and set it as the
index. Expect a DataFrame indexed by date with one column per ticker (AAPL, MSFT, IBM).

In [ ]:
prices = pd.read_csv("../data/closing_price.csv", parse_dates=["Date"], index_col="Date")
prices.head(3)

Call `.hvplot()` on the whole DataFrame with no arguments, the same call as plain `.plot()` in
Activity 1. Expect one interactive line per ticker, all on the same axes, with a legend you can
click to hide or show a line, and a hover tooltip that reports the exact price on the date under
your cursor.

In [ ]:
prices.hvplot()

Give each ticker its own panel with `subplots=True`, then stack the panels vertically with
`.cols(1)`. Expect three independent interactive panels, one per ticker, each still zoomable and
hoverable on its own.

In [ ]:
prices.hvplot(subplots=True).cols(1)

## A widget for one keyword

Every hvPlot call so far has produced a fixed chart. Passing `groupby=` produces a dropdown above
the chart instead: pick a value, and the chart redraws in the browser without rerunning the cell.

Build a small frame the widget can work with first: keep only the 5 most common DRG
(diagnosis-related group) definitions in the data, then average the ratio by DRG and state. Pass
`groupby="DRG Definition"` to `.hvplot.bar` so the chart gets a dropdown control. Expect a dropdown
above the chart listing five DRG definitions; selecting one redraws the bars without rerunning the
cell.

In [ ]:
top_drgs = claims[claims["DRG Definition"].isin(claims["DRG Definition"].value_counts().head(5).index)]
state_drg = top_drgs.groupby(["DRG Definition", "Provider State"])["ratio"].mean().reset_index()
state_drg.hvplot.bar(x="Provider State", y="ratio", groupby="DRG Definition", rot=90, height=400, width=900)

## A third library, for this afternoon

hvPlot is not the only interactive option in the Python ecosystem. Plotly Express (`plotly.express`,
usually imported as `px`) builds the same kind of interactive charts with a different, widely used
syntax. It matters today because this afternoon's Streamlit app renders Plotly figures natively with
`st.plotly_chart`, so the chart you write here carries over almost unchanged.

Rebuild the same state bar chart in Plotly Express. `px.bar` needs the state names as an actual
column rather than an index, so call `.reset_index()` on `by_state` first. Expect the same
interactive bar chart, now built with Plotly's toolbar and hover style instead of bokeh's.

In [ ]:
import plotly.express as px

px.bar(
    by_state.head(15).reset_index(),
    x="Provider State",
    y="ratio",
    title="New Jersey hospitals bill 6.7 times what Medicare pays",
    labels={"ratio": "Average charge to payment ratio"},
)

Streamlit has a dedicated `st.plotly_chart` function that takes a Plotly figure and renders it
directly in an app, with the same hover and zoom behavior you just saw. hvPlot figures need extra
steps to embed in Streamlit, so the rest of the course uses Plotly for anything that ends up inside
a Streamlit app. Keep this exact `px.bar` pattern, you will reuse it almost unchanged this
afternoon.

## Your turn

Two exercises. Reuse the patterns from this notebook.

**Exercise 1.** In Activity 1 you built a scatter of `Average Covered Charges` against
`Average Total Payments` with plain `.plot`. Rebuild it as an interactive chart:

1. Take a random sample of 3000 rows from `claims` with `random_state=7`.
   Why sample first: with 163065 rows, an interactive scatter has to turn every single point into
   its own hoverable, zoomable element, and a browser drawing all of them at once gets sluggish. A
   static image can compress all 163065 points into pixels; an interactive one cannot. Thinning the
   data for an interactive view is a normal, honest engineering step, not a shortcut.
2. Call `.hvplot.scatter` on the sample, with `x="Average Total Payments"`,
   `y="Average Covered Charges"`, and add `hover_cols=["Provider Name", "Provider State"]` so the
   tooltip names the hospital.
3. Hover the point furthest from the rest of the cloud and read its `Provider Name` and
   `Provider State` off the tooltip.

Expected: a scatter of about 3000 points, most clustered together with a handful stretching far to
the upper right. Hovering the furthest point shows its provider name and state in the tooltip,
which a static image never could.

In [ ]:
sample = claims.sample(3000, random_state=7)
sample.hvplot.scatter(
    x="Average Total Payments",
    y="Average Covered Charges",
    hover_cols=["Provider Name", "Provider State"],
    title="A minority of providers bill far above what Medicare pays",
    height=450,
    width=900,
    alpha=0.4,
)

**Exercise 2.** In Activity 1 you normalized the closing prices so all three tickers start at 1.0,
then plotted them with plain `.plot`. Rebuild that normalized chart with `.hvplot()` instead:
divide `prices` by its first row (`prices.div(prices.iloc[0])`) to normalize it, then call
`.hvplot()` on the result. `prices` is still loaded from earlier in this notebook.

Then add one comment line stating one thing the interactive version lets you do that the static
image did not.

Expected: three lines that all start at 1.0 and diverge from there, now with hover and zoom.

In [ ]:
normalized = prices.div(prices.iloc[0])
normalized.hvplot(
    title="Normalized to the first trading day, all three start at 1.0",
    ylabel="Growth relative to day one",
    xlabel="Date",
    height=450,
    width=900,
)
# One thing this lets you do that the static chart did not: zoom into a single stretch of dates
# and hover a line to read the exact growth value on that day.